In [1]:
import pandas as pd
import numpy as np

### Load dataset into DataFrame

In [2]:
df = pd.read_csv("./Inputs/BRK.csv")

## 1. Check Data Structure

### Display first 5 rows in the DataFrame

In [3]:
df.head()

,date,open,high,low,close,adj_close,volume
0,NaN,BRK-B,BRK-B,BRK-B,BRK-B,BRK-B,BRK-B
1,1996-05-09,22.200000762939453,24.399999618530273,22.200000762939453,23.200000762939453,23.200000762939453,4290000
2,1996-05-10,24.0,24.200000762939453,23.600000381469727,24.0,24.0,1060000
3,1996-05-13,24.0,24.100000381469727,23.299999237060547,23.899999618530273,23.899999618530273,700000
4,1996-05-14,24.0,24.100000381469727,23.100000381469727,23.600000381469727,23.600000381469727,310000


### Display last five rows in the DataFrame

In [4]:
df.tail()

,date,open,high,low,close,adj_close,volume
7358,2025-08-06,465.0,469.54998779296875,463.0,468.9100036621094,468.9100036621094,4557100
7359,2025-08-07,469.20001220703125,472.70001220703125,461.3699951171875,461.4700012207031,461.4700012207031,5755700
7360,2025-08-08,462.94000244140625,465.8299865722656,462.54998779296875,465.3999938964844,465.3999938964844,3330600
7361,2025-08-11,466.4599914550781,468.489990234375,463.5,464.7300109863281,464.7300109863281,3712100
7362,2025-08-12,465.5299987792969,472.4700012207031,465.05999755859375,470.3900146484375,470.3900146484375,3767300


### Check that the DataFrame is sorted by dates in ascending order

In [5]:
if df['date'].is_monotonic_increasing == False:
    print("DataFrame is sorted by dates in ascending order ")

DataFrame is sorted by dates in ascending order 


## 2. Validate the Column's format

We are expecting to see datetime64 for date column and numeric formats for OHLC and ADJ Close columns.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7363 entries, 0 to 7362
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   date       7362 non-null   object
 1   open       7363 non-null   object
 2   high       7363 non-null   object
 3   low        7363 non-null   object
 4   close      7363 non-null   object
 5   adj_close  7363 non-null   object
 6   volume     7363 non-null   object
dtypes: object(7)
memory usage: 402.8+ KB


In [7]:
# Columns
date_col = 'date'
price_cols = ['open', 'high', 'low', 'close', 'adj_close', 'volume']

# 1️⃣ Parse date
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

# 2️⃣ Convert numeric columns
df[price_cols] = df[price_cols].apply(pd.to_numeric, errors='coerce')

# 3️⃣ Drop rows with invalid date or numbers
df = df.dropna(subset=[date_col] + price_cols)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7362 entries, 1 to 7362
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       7362 non-null   datetime64[ns]
 1   open       7362 non-null   float64       
 2   high       7362 non-null   float64       
 3   low        7362 non-null   float64       
 4   close      7362 non-null   float64       
 5   adj_close  7362 non-null   float64       
 6   volume     7362 non-null   float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 460.1 KB


In [9]:
df.tail()

,date,open,high,low,close,adj_close,volume
7358,2025-08-06,465.000000,469.549988,463.000000,468.910004,468.910004,4557100.0
7359,2025-08-07,469.200012,472.700012,461.369995,461.470001,461.470001,5755700.0
7360,2025-08-08,462.940002,465.829987,462.549988,465.399994,465.399994,3330600.0
7361,2025-08-11,466.459991,468.489990,463.500000,464.730011,464.730011,3712100.0
7362,2025-08-12,465.529999,472.470001,465.059998,470.390015,470.390015,3767300.0


## 3. Look for Missing or Invalid Values

Check for missing values.

In [10]:
df.isna().sum()

date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

All the columns have values in the their corresponding cells. Check Passed.

---

Check for cells with blank strings in them

In [11]:
(df == '').sum()

date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

None of the columns have blank strings in their corresponding cells. Check Passed.

---

Check for columns that must contain numeric values, but instead have non-numeric values in them.

In [12]:
df[['open','high','low','close','adj_close','volume']].apply(pd.to_numeric, errors='coerce').isna().sum()

open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

None of the columns non-numeric values in their corresponding cells. Check Passed.

In [13]:
bad_rows = df[
    df[['open','high','low','close','adj_close','volume']]
    .apply(pd.to_numeric, errors='coerce')
    .isna()
    .any(axis=1)
]

bad_rows


,date,open,high,low,close,adj_close,volume


In [14]:
df = df.dropna(subset=['open','high','low','close','adj_close','volume'])


In [15]:
df[['open','high','low','close','adj_close','volume']].apply(pd.to_numeric, errors='coerce').isna().sum()

open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

---

Check for negative prices in OHLC and ADJ Close columns. We should not have any negative numebrs in them.

In [17]:
(df[['open','high','low','close','adj_close']] < 0).sum()

open         0
high         0
low          0
close        0
adj_close    0
dtype: int64

None of the cells in OHLC and ADJ Close columns have prices that are less than zero. Check Passed.

---

Check for zero prices in OHLC and ADJ Close columns. We should not have any zero prices in them.

In [18]:
(df[['open','high','low','close','adj_close']] == 0).sum()

open         0
high         0
low          0
close        0
adj_close    0
dtype: int64

None of the prices in OHLC and ADJ Close columns have zero prices in them. Check Passed. 

---

Check for negative or zero volume.

In [19]:
(df['volume'] <= 0).sum()

np.int64(0)

We found one entry that with volume cell that is less than or equal to zero. Let's find this entry and analyze it.

In [20]:
df[df['volume'] <= 0]

,date,open,high,low,close,adj_close,volume


It seems that Apple stock did not trade on 1981-08-10 as OHLC are the same and the volume is 0. Therefore we are going to drop this row.

## 4. Consistency Checks

We are going to check:
1.  "Low" prices are always lower than "Open" prices.
2.  "Open" Prices are always lower than "High" prices.
3.  "Low" Prices are always lower than "Close" prices.
4.  "Close" Prices are always lower than "High" Prices. 

In [21]:
invalid_ohlc = df[
    (df['low'] > df['open']) |
    (df['open'] > df['high']) |
    (df['low'] > df['close']) |
    (df['close'] > df['high'])
]

invalid_ohlc

,date,open,high,low,close,adj_close,volume


All the prices are valid. Check Passed.

## 5. Detect Duplicated Rows

We are going to check if there are any duplicate rows.

In [22]:
df.duplicated().sum()

np.int64(0)

DataFrame does not have any duplicate rows. Check Passed.

---

### 7. Clear DataFrame and save it into new CSV file

In [23]:
# Drop entry with volume less than or equal to zero.
df_cleaned = df[df['volume'] > 0].copy()

# Reset index
df_cleaned.reset_index(drop=True, inplace=True)

# Ensure that there are no entries with volume less than or equal to zero.
(df_cleaned['volume'] <= 0).sum()

np.int64(0)

In [24]:
# Save cleaned DataFrame into CSV file
df_cleaned.to_csv("./Outputs/BRK_Pre_Processed.csv", index=False)